In [ ]:
!git clone https://github.com/facebookresearch/MetaCLIP.git

Cloning into 'MetaCLIP'...
remote: Enumerating objects: 711, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 711 (delta 122), reused 150 (delta 80), pack-reused 486 (from 2)
Receiving objects: 100% (711/711), 25.62 MiB | 23.15 MiB/s, done.
Resolving deltas: 100% (284/284), done.


In [ ]:
!pip install -qq openclip-torch

ERROR: Could not find a version that satisfies the requirement openclip-torch (from versions: none)
ERROR: No matching distribution found for openclip-torch


In [ ]:
!pip install -qq ftfy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00


In [ ]:
%cd MetaCLIP

/content/MetaCLIP


In [ ]:
import torch
from PIL import Image
from src.open_clip.factory import create_model_and_transforms, list_models
from src.open_clip.tokenizer import tokenize


# model, _, preprocess = create_model_and_transforms('ViT-B-32-quickgelu', pretrained='metaclip_400m')  # for 2.5B use 'metaclip_fullcc' in OpenCLIP or 'metaclip_2_5b' in this repo

# image = preprocess(Image.open("/content/dog.jpeg")).unsqueeze(0)
# text = tokenize(["a diagram", "a dog", "a cat"])

# with torch.no_grad():
#     image_features = model.encode_image(image)
#     text_features = model.encode_text(text)
#     image_features /= image_features.norm(dim=-1, keepdim=True)
#     text_features /= text_features.norm(dim=-1, keepdim=True)

#     text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

# print("Label probs:", text_probs)

In [ ]:
from src.open_clip.pretrained import list_pretrained
list_pretrained()

[('RN50', 'openai'),
 ('RN50', 'yfcc15m'),
 ('RN50', 'cc12m'),
 ('RN50-quickgelu', 'openai'),
 ('RN50-quickgelu', 'yfcc15m'),
 ('RN50-quickgelu', 'cc12m'),
 ('RN101', 'openai'),
 ('RN101', 'yfcc15m'),
 ('RN101-quickgelu', 'openai'),
 ('RN101-quickgelu', 'yfcc15m'),
 ('RN50x4', 'openai'),
 ('RN50x16', 'openai'),
 ('RN50x64', 'openai'),
 ('ViT-B-32', 'openai'),
 ('ViT-B-32', 'laion2b_e16'),
 ('ViT-B-32', 'laion400m_e31'),
 ('ViT-B-32', 'laion400m_e32'),
 ('ViT-B-32-quickgelu', 'openai'),
 ('ViT-B-32-quickgelu', 'laion400m_e31'),
 ('ViT-B-32-quickgelu', 'laion400m_e32'),
 ('ViT-B-32-quickgelu', 'metaclip_400m'),
 ('ViT-B-32-quickgelu', 'metaclip_2_5b'),
 ('ViT-B-32-quickgelu', 'metaclip_fullcc'),
 ('ViT-B-32-quickgelu', 'metaclip400m'),
 ('ViT-B-32-quickgelu', 'metaclip2_5b'),
 ('ViT-B-16', 'openai'),
 ('ViT-B-16', 'laion400m_e31'),
 ('ViT-B-16', 'laion400m_e32'),
 ('ViT-B-16-quickgelu', 'metaclip_400m'),
 ('ViT-B-16-quickgelu', 'metaclip_2_5b'),
 ('ViT-B-16-quickgelu', 'metaclip_fullcc')

In [ ]:
pretrained_models = list_pretrained()  # Call the function to get the list of models
len(pretrained_models)  # Now, get the length of the list

51

In [ ]:
pretrained_models[47]

('ViT-H-14', 'metaclip_v1_2_altogether')

In [ ]:
modelslist=list_pretrained()
sizes1=[224,224,224,224,224,224,224,224,224,288,384,224,224,224,224,224,224,224,224,224,224,224,224,224,240,240,224,224,224,224,336,224,224,224,224,224, 224, 224,224,336,336, 336, 336]
sizes2=[224,224,224,224,224,224,224,224,224,288,384,224,224,224,224,224,224,224,224,224,224,224,224,224,240,240,224,224,224,224,336,224,224,224,224,224, 224, 224,224,224,224,336,224, 224]
sizes=sizes1+sizes2+sizes2+sizes2

In [ ]:
import torch
cuda = torch.device('cuda')
device = torch.device("cuda")
#from cadence models
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!unzip -qq /content/gdrive/MyDrive/datasets/8nadataisic.zip -d /content/ddos3/

replace /content/ddos3/Natural Data-augmentation for skin Lesions (ISIC-2017 Challenge dataset)/Kert/ISIC_0012086_1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import os
import torch
import gc
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import scipy.io as sio
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
#model, preprocess = clip.load('RN50', device)

def get_features(dataset,modelname1,modelname2):
    dataloader1 = torch.utils.data.DataLoader(dataset, batch_size=200)
    all_features = []
    all_labels = []
    model, train_transform, eval_transform = create_model_and_transforms(modelname1, pretrained=modelname2)
    model.cuda()
    model.eval()
    with torch.no_grad():
        for (batch_idx, batch) in enumerate(dataloader1, 0):
            images, labels = batch
            images = images.to(device)
            model = model.half().to(device)
            images = images.half().to(device)
            features = model.encode_image(images.to(device))
            all_features.append(features)
            all_labels.append(labels)
        featfinal=torch.cat(all_features).cpu().numpy()
        labfinal= torch.cat(all_labels).cpu().numpy()
        final=np.column_stack((featfinal,labfinal))
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return final

# # Calculate the image features
def getdataset(siz):
    transform = transforms.Compose([transforms.Resize(siz),
                                transforms.CenterCrop(siz),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])])

    dataset = datasets.ImageFolder("/content/ddos3/Natural Data-augmentation for skin Lesions (ISIC-2017 Challenge dataset)", transform = transform)
    return dataset
# Access dataset.classes after calling getdataset
dataset = getdataset(224) # Assuming a default size of 224
dataset.classes  # Now, access dataset.classes

['Kert', 'Melanoma', 'Nevi']

In [ ]:
sizes[30]=224
sizes[33]=256
sizes[39]=224
sizes[40]=224
sizes[41]=224
sizes[42]=224
sizes[52]=224
sizes[53]=224
sizes[54]=224
sizes[55]=224
sizes[56]=224
sizes[9]=224
sizes[10]=288
sizes[11]=384
sizes[12]=448
sizes[13]=224
sizes[100]=224
sizes[106]=224
sizes[101]=224
sizes[102]=256
sizes[103]=256
sizes[112]=336

sizes[109]=336
sizes[110]=336
sizes[104]=384
sizes[54]=240
sizes[55]=240
sizes[56]=224
sizes[61]=224
sizes[62]=224
sizes[63]=378
sizes[67]=336
sizes[68]=224
sizes[71]=378
sizes[73]=224
sizes[96]=224
sizes[97]=224
sizes[98]=336

In [ ]:
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
#49..30G

In [ ]:
for modnum in range(47, 48):
      modnew=modelslist[modnum]
      final = get_features(getdataset(sizes[modnum]), modnew[0],modnew[1])
      filepath='/content/gdrive/MyDrive/skinmat2/'
      filename1=modnew[0]+modnew[1]
      fileext='skisic.mat'
      filefinal=filepath+filename1+fileext
      sio.savemat(filefinal,{'final':final})
      torch.cuda.empty_cache()
      print('Model No=',modnum)

100%|█████████████████████████████████████| 3.94G/3.94G [02:26<00:00, 26.9MiB/s]


Model No= 47
